# PatchCraft SVM Inference

## Library

In [ ]:
from PIL import Image, ImageFilter
import numpy as np
import cv2

import random
import math
import pickle
import concurrent.futures

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.transforms import transforms

In [ ]:
device = torch.device("cpu")

## Smash&Reconstruction

In [ ]:
class PatchGenerator:
    def img_to_patches(self, img, patch_size):
        if img.size != (256,256):
            img = img.resize(size=(256,256))

        patches = []
        for i in range(0, img.height, patch_size):
            for j in range(0, img.width, patch_size):
                box = (j, i, (j + patch_size), (i + patch_size))
                patch = np.asarray(img.crop(box))
                patches.append(patch)
        return patches


    def get_l1(self, v, x, y):
        l1 = 0
        for i in range(0, y-1):
            for j in range(0, x):
                l1 += abs(v[j][i]-v[j][i+1])
        return l1

    def get_l2(self, v, x, y):
        l2 = 0
        for i in range(0, y):
            for j  in range(0, x-1):
                l2 += abs(v[j][i]-v[j+1][i])
        return l2

    def get_l3l4(self, v, x, y):
        l3 = l4 = 0
        for i in range(0, y-1):
            for j  in range(0, x-1):
                l3 += abs(v[j][i]-v[j+1][i+1])
                l4 += abs(v[j+1][i]-v[j][i+1])
        return l3 + l4

    def get_patch_IPV(self, patch):
        x, y = patch.shape
        l1 = l2 = l3l4 = 0
        with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor :
            future_l1 = executor.submit(self.get_l1, patch, x, y)
            future_l2 = executor.submit(self.get_l2, patch, x, y)
            future_l3l4 = executor.submit(self.get_l3l4, patch, x, y)

            l1 = future_l1.result()
            l2 = future_l2.result()
            l3l4 = future_l3l4.result()
        return  l1 + l2 + l3l4


    def extract_rich_and_poor_textures(self, pixel_var_degrees, patches):
        patch_var_degree_pair = []
        for i, j in enumerate(pixel_var_degrees) :
            patch_var_degree_pair.append({'patch': patches[i], 'pixel_var_degree': j})
        patch_var_degree_pair.sort(key=lambda pair: pair['pixel_var_degree'])

        mid = len(patch_var_degree_pair) // 2
        rich_texture_patches = [pair['patch'] for pair in patch_var_degree_pair[mid:]]
        poor_texture_patches = [pair['patch'] for pair in patch_var_degree_pair[:mid]]

        return rich_texture_patches, poor_texture_patches


    def get_complete_image(self, patches, patch_size):
        random.shuffle(patches)
        subimg_dim = 192
        subimg_len = subimg_dim // patch_size
        target_patches = math.pow(subimg_len, 2)

        p_len = len(patches)
        while len(patches) < target_patches:
            patches.append(patches[random.randint(0, p_len-1)])

        grid = np.asarray(patches).reshape((subimg_len, subimg_len, patch_size, patch_size, 3))
        rows = []
        for i in range(subimg_len) :
            rows.append(np.concatenate(grid[i,:], axis=1))
        img = np.concatenate(rows, axis=0)

        return img


    def smash_n_reconstruct(self, img, patch_size=32):
        patches = self.img_to_patches(img, patch_size)

        pixel_var_degrees = []
        for patch in patches :
            grayscale_patch = cv2.cvtColor(patch, cv2.COLOR_RGB2GRAY).astype(np.int32)
            pixel_var_degrees.append(self.get_patch_IPV(grayscale_patch))

        r_patch, p_patch = self.extract_rich_and_poor_textures(pixel_var_degrees, patches)

        rich_texture, poor_texture = None, None
        with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
            rich_texture_future = executor.submit(self.get_complete_image, r_patch, patch_size)
            poor_texture_future = executor.submit(self.get_complete_image, p_patch, patch_size)

            rich_texture = rich_texture_future.result()
            poor_texture = poor_texture_future.result()

        return rich_texture, poor_texture


# Create PatchGenerator interface for Smash & Reconstruction
smash_n_reconstruct = PatchGenerator().smash_n_reconstruct

## High Pass Filter

In [ ]:
filter_class_1 = [
    np.array([
        [1, 0, 0],
        [0, -1, 0],
        [0, 0, 0]
    ], dtype=np.float32),
    np.array([
        [0, 1, 0],
        [0, -1, 0],
        [0, 0, 0]
    ], dtype=np.float32),
    np.array([
        [0, 0, 1],
        [0, -1, 0],
        [0, 0, 0]
    ], dtype=np.float32),
    np.array([
        [0, 0, 0],
        [1, -1, 0],
        [0, 0, 0]
    ], dtype=np.float32),
    np.array([
        [0, 0, 0],
        [0, -1, 1],
        [0, 0, 0]
    ], dtype=np.float32),
    np.array([
        [0, 0, 0],
        [0, -1, 0],
        [1, 0, 0]
    ], dtype=np.float32),
    np.array([
        [0, 0, 0],
        [0, -1, 0],
        [0, 1, 0]
    ], dtype=np.float32),
    np.array([
        [0, 0, 0],
        [0, -1, 0],
        [0, 0, 1]
    ], dtype=np.float32)
]

filter_class_2 = [
    np.array([
        [1, 0, 0],
        [0, -2, 0],
        [0, 0, 1]
    ], dtype=np.float32),
    np.array([
        [0, 1, 0],
        [0, -2, 0],
        [0, 1, 0]
    ], dtype=np.float32),
    np.array([
        [0, 0, 1],
        [0, -2, 0],
        [1, 0, 0]
    ], dtype=np.float32),
    np.array([
        [0, 0, 0],
        [1, -2, 1],
        [0, 0, 0]
    ], dtype=np.float32),
]

filter_class_3 = [
    np.array([
        [-1, 0, 0, 0, 0],
        [0, 3, 0, 0, 0],
        [0, 0, -3, 0, 0],
        [0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0]
    ], dtype=np.float32),
    np.array([
        [0, 0, -1, 0, 0],
        [0, 0, 3, 0, 0],
        [0, 0, -3, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0]
    ], dtype=np.float32),
    np.array([
        [0, 0, 0, 0, -1],
        [0, 0, 0, 3, 0],
        [0, 0, -3, 0, 0],
        [0, 1, 0, 0, 0],
        [0, 0, 0, 0, 0]
    ], dtype=np.float32),
    np.array([
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 1, -3, 3, -1],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0]
    ], dtype=np.float32),
    np.array([
        [0, 0, 0, 0, 0],
        [0, 1, 0, 0, 0],
        [0, 0, -3, 0, 0],
        [0, 0, 0, 3, 0],
        [0, 0, 0, 0, -1]
    ], dtype=np.float32),
    np.array([
        [0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, -3, 0, 0],
        [0, 0, 3, 0, 0],
        [0, 0, -1, 0, 0]
    ], dtype=np.float32),
    np.array([
        [0, 0, 0, 0, 0],
        [0, 0, 0, 1, 0],
        [0, 0, -3, 0, 0],
        [0, 3, 0, 0, 0],
        [-1, 0, 0, 0, 0]
    ], dtype=np.float32),
    np.array([
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [-1, 3, -3, 1, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0]
    ], dtype=np.float32)
]

filter_edge_3x3 = [
    np.array([
        [-1, 2, -1],
        [2, -4, 2],
        [0, 0, 0]
    ], dtype=np.float32),
    np.array([
        [0, 2, -1],
        [0, -4, 2],
        [0, 2, -1]
    ], dtype=np.float32),
    np.array([
        [0, 0, 0],
        [2, -4, 2],
        [-1, 2, -1]
    ], dtype=np.float32),
    np.array([
        [-1, 2, 0],
        [2, -4, 0],
        [-1, 2, 0]
    ], dtype=np.float32),
]

filter_edge_5x5 = [
    np.array([
        [-1, 2, -2, 2, -1],
        [2, -6, 8, -6, 2],
        [-2, 8, -12, 8, -2],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0]
    ], dtype=np.float32),
    np.array([
        [0, 0, -2, 2, -1],
        [0, 0, 8, -6, 2],
        [0, 0, -12, 8, -2],
        [0, 0, 8, -6, 2],
        [0, 0, -2, 2, -1]
    ], dtype=np.float32),
    np.array([
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0],
        [-2, 8, -12, 8, -2],
        [2, -6, 8, -6, 2],
        [-1, 2, -2, 2, -1]
    ], dtype=np.float32),
    np.array([
        [-1, 2, -2, 0, 0],
        [2, -6, 8, 0, 0],
        [-2, 8, -12, 0, 0],
        [2, -6, 8, 0, 0],
        [-1, 2, -2, 0, 0]
    ], dtype=np.float32),
]

square_3x3 = np.array([
    [-1, 2, -1],
    [2, -4, 2],
    [-1, 2, -1]
], dtype=np.float32)

square_5x5 = np.array([
    [-1, 2, -2, 2, -1],
    [2, -6, 8, -6, 2],
    [-2, 8, -12, 8, -2],
    [2, -6, 8, -6, 2],
    [-1, 2, -2, 2, -1]
], dtype=np.float32)

normalized_filter_class_2   =   [hpf / 2 for hpf in filter_class_2]
normalized_filter_class_3   =   [hpf / 3 for hpf in filter_class_3]
normalized_filter_edge_3x3  =   [hpf / 4 for hpf in filter_edge_3x3]
normalized_square_3x3       =   square_3x3 / 4
normalized_filter_edge_5x5  =   [hpf / 12 for hpf in filter_edge_5x5]
normalized_square_5x5       =   square_5x5 / 12

all_normalized_hpf_list = filter_class_1 + normalized_filter_class_2 + normalized_filter_class_3 + \
                            normalized_filter_edge_3x3 + normalized_filter_edge_5x5 + \
                            [normalized_square_3x3, normalized_square_5x5]

In [ ]:
class HPF:
    def __init__(self):
        all_hpf_list_5x5 = []

        for hpf in all_normalized_hpf_list:
            if hpf.shape[0] == 3:
                hpf = np.pad(hpf, pad_width=((1, 1), (1, 1)), mode='constant')
            all_hpf_list_5x5.append(hpf)

        hpf_weight = torch.from_numpy(np.array(all_hpf_list_5x5)).view(30, 1, 5, 5).contiguous()
        hpf_weight = torch.nn.Parameter(hpf_weight.repeat(1, 3, 1, 1), requires_grad=False)

        self.hpf = nn.Conv2d(3, 30, kernel_size=5, padding=2, bias=False)
        self.hpf.weight = hpf_weight

    def apply_all_filters(self, input):
        output = self.hpf(input)
        return output

apply_all_filters = HPF().apply_all_filters

## Preprocessing Wrapper

In [ ]:
def ndarrimg2tensor(arr):
    tensor = torch.from_numpy(arr)
    tensor = tensor.permute(2, 0, 1)
    tensor = tensor.to(torch.float32)
    return tensor

In [ ]:
def normalize(tensor):
    transform_normalize = transforms.Compose([
        transforms.Normalize(mean=[125.9, 118.5, 109.5], std=[73.1, 72, 74])
    ])
    normalized_tensor = transform_normalize(tensor)
    return normalized_tensor

In [ ]:
def preprocess(img, patch_size=32):
    rt, pt = smash_n_reconstruct(img, patch_size)

    rt_tensor = ndarrimg2tensor(rt)
    pt_tensor = ndarrimg2tensor(pt)

    normalized_rt = normalize(rt_tensor)
    normalized_pt = normalize(pt_tensor)

    frt = apply_all_filters(rt_tensor)
    fpt = apply_all_filters(pt_tensor)

    return frt, fpt

## Feature Extraction Model

In [ ]:
class ConvoBlock (nn.Module):
  def __init__(self, in_channels, out_channels, kernel_size):
    super(ConvoBlock, self).__init__()
    self.conv = nn.Conv2d(in_channels, out_channels, kernel_size)
    self.bn = nn.BatchNorm2d(out_channels)

  def forward(self, input):
    x = self.conv(input)
    x = F.relu(x)
    x = self.bn(x)
    return x

In [ ]:
class PatchCraft(nn.Module):
    def __init__(self):
        super(PatchCraft, self).__init__()

        self.conv_block_rich = ConvoBlock(30, 32, 3)
        self.conv_block_poor = ConvoBlock(30, 32, 3)

        self.conv_block = ConvoBlock(32, 32, 3)

        self.conv_blocks_1 = nn.ModuleList()
        for i in range(3):
            self.conv_blocks_1.append(ConvoBlock(32, 32, 3))
        self.bn_1 = nn.BatchNorm2d(32)

        self.conv_blocks_2 = nn.ModuleList()
        for i in range(4):
            self.conv_blocks_2.append(ConvoBlock(32, 32, 3))
        self.avgpool_1 = nn.AvgPool2d(2)

        self.conv_blocks_3 = nn.ModuleList()
        for i in range(2):
            self.conv_blocks_3.append(ConvoBlock(32, 32, 3))
        self.avgpool_2 = nn.AvgPool2d(2)

        self.conv_blocks_4 = nn.ModuleList()
        for i in range(2):
            self.conv_blocks_4.append(ConvoBlock(32, 32, 3))
        self.globpool = nn.AdaptiveAvgPool2d((1,1))

        self.flatten = nn.Flatten()
        self.clf = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x_rich, x_poor):

        x1 = self.conv_block_rich(x_rich)
        x1 = F.hardtanh(x1, min_val=-1, max_val=1)

        x2 = self.conv_block_poor(x_poor)
        x2 = F.hardtanh(x2, min_val=-1, max_val=1)

        x = x1-x2

        x = self.conv_block(x)

        for block in self.conv_blocks_1:
            x = block(x)
        x = self.bn_1(x)

        for block in self.conv_blocks_2:
            x = block(x)
        x = self.avgpool_1(x)

        for block in self.conv_blocks_3:
            x = block(x)
        x = self.avgpool_2(x)

        for block in self.conv_blocks_4:
            x = block(x)
        x = self.globpool(x)

        x = self.flatten(x)
        x = self.clf(x)
        x = self.sigmoid(x)

        return x

In [ ]:
class HeadlessPatchCraft(nn.Module):
    def __init__(self, pretrained_model):
        super(HeadlessPatchCraft, self).__init__()

        self.conv_block_rich = pretrained_model.conv_block_rich
        self.conv_block_poor = pretrained_model.conv_block_poor

        self.conv_block = pretrained_model.conv_block

        self.conv_blocks_1 = pretrained_model.conv_blocks_1
        self.bn_1 = pretrained_model.bn_1

        self.conv_blocks_2 = pretrained_model.conv_blocks_2
        self.avgpool_1 = pretrained_model.avgpool_1

        self.conv_blocks_3 = pretrained_model.conv_blocks_3
        self.avgpool_2 = pretrained_model.avgpool_2

        self.conv_blocks_4 = pretrained_model.conv_blocks_4
        self.globpool = pretrained_model.globpool

        self.flatten = pretrained_model.flatten

    def forward(self, x_rich, x_poor):

        x1 = self.conv_block_rich(x_rich)
        x1 = F.hardtanh(x1, min_val=-1, max_val=1)

        x2 = self.conv_block_poor(x_poor)
        x2 = F.hardtanh(x2, min_val=-1, max_val=1)

        x = x1-x2

        x = self.conv_block(x)

        for block in self.conv_blocks_1:
            x = block(x)
        x = self.bn_1(x)

        for block in self.conv_blocks_2:
            x = block(x)
        x = self.avgpool_1(x)

        for block in self.conv_blocks_3:
            x = block(x)
        x = self.avgpool_2(x)

        for block in self.conv_blocks_4:
            x = block(x)
        x = self.globpool(x)

        x = self.flatten(x)
        return x

## Inference

In [32]:
class ModelInference:
    def __init__(self, feature_extractor_path, svm_path):
        device = torch.device("cpu")
        weight = torch.load(feature_extractor_path, map_location=device, weights_only=True)

        self.feature_extractor = HeadlessPatchCraft(PatchCraft())
        self.feature_extractor.load_state_dict(weight)
        self.feature_extractor = self.feature_extractor.to(device)

        with open(svm_path, 'rb') as file:
            self.best_clf = pickle.load(file)


    def multiple_crop(self, input_path):
        img = Image.open(input_path).convert("RGB")

        w, h = img.width, img.height
        max_d = max(w, h)
        min_d = min(w, h)
        n = math.ceil(max_d / min_d)
        offset = 0 if n == 1 else (max_d-min_d) / (n-1)

        crop_imgs = []
        if w > h :
            for i in range(n) :
                box = (int(i*offset), 0, int(i*offset+min_d), min_d)
                crop_imgs.append(img.crop(box))
        else :
            for i in range(n) :
                box = (0, int(i*offset), min_d, int(i*offset+min_d))
                crop_imgs.append(img.crop(box))

        return crop_imgs


    def inference(self, input_path):
        crop_imgs = self.multiple_crop(input_path)
        features = []
        preds = []

        for crop in crop_imgs :
            frt, fpt = preprocess(crop)

            feature = self.feature_extractor(frt.unsqueeze(0), fpt.unsqueeze(0))
            feature = feature.detach().numpy()
            features.append(feature)

            pred = self.best_clf.predict(feature)
            preds.append(pred)

        result = np.mean(preds)

        if result == 0.5 :
            probs = []
            for feature in features :
                prob = self.best_clf.predict_proba(feature)
                probs.append(prob)

            proba = np.mean(probs,axis=0)
            result = 0 if proba[0][0] > proba[0][1] else 1

        return int(result)

In [33]:
feature_extractor_path = '/content/headless_model.pth'
svm_path = 'svm.pkl'

inference = ModelInference(feature_extractor_path, svm_path).inference

In [34]:
img_path = 'dalle3_22.png'
result = inference(img_path)
print(f'Prediction : {'fake' if result == 1 else 'real'}')

Prediction : fake
